# Crop Doctor – Vision Model Training & ExecuTorch Export
**Hardware Target:** Google Colab Free GPU (NVIDIA T4) | **Device Deployment:** Arm Cortex-A (6 GB RAM)

This notebook trains a production-grade **MobileNetV3-Large** vision classifier across the 25 Crop Doctor target classes, performs temperature calibration, audits real-field holdout accuracy, and exports directly to **ExecuTorch (`vision.pte`)** for on-device Arm inference.

In [ ]:
# ── 1. Install Dependencies & ExecuTorch ──────────────────────────────────────
!pip install -q timm scikit-learn matplotlib pyyaml
!pip install -q executorch torchvision torch

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

In [ ]:
# ── 2. Mount Google Drive & Extract Dataset ──────────────────────────────────
from google.colab import drive
import os, glob, sys

drive.mount('/content/drive')

DATA_DIR = '/content/crop_data'
os.makedirs(DATA_DIR, exist_ok=True)

# Search Google Drive root and subfolders for dataset archives
candidate_tars = glob.glob('/content/drive/MyDrive/**/crop_doctor_dataset*.tar', recursive=True) + glob.glob('/content/drive/MyDrive/crop*.tar')
candidate_zips = glob.glob('/content/drive/MyDrive/**/crop_doctor_dataset*.zip', recursive=True) + glob.glob('/content/drive/MyDrive/crop*.zip')

if candidate_tars:
    tar_path = candidate_tars[0]
    print(f"Found tar dataset at: {tar_path} ({os.path.getsize(tar_path)/(1024*1024):.1f} MB)")
    print("Extracting dataset to local Colab SSD... ")
    !tar -xf "{tar_path}" -C {DATA_DIR}
    class_dirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"✓ Extracted successfully! Total classes found: {len(class_dirs)}")
elif candidate_zips:
    zip_path = candidate_zips[0]
    print(f"Found zip dataset at: {zip_path}")
    print("Extracting dataset to local Colab SSD... ")
    !unzip -q "{zip_path}" -d {DATA_DIR}
    class_dirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"✓ Extracted successfully! Total classes found: {len(class_dirs)}")
else:
    print("[ERROR] crop_doctor_dataset.tar was not found in your Google Drive!")
    print("Here are the files currently in your Google Drive root:")
    !ls -lh /content/drive/MyDrive/


In [ ]:
# ── 3. Clone / Update Crop Doctor Repository ─────────────────────────────────
!git clone https://github.com/varun-gajula/crop-doctor-offline.git /content/crop-doctor-offline 2>/dev/null || (cd /content/crop-doctor-offline && git pull)
if '/content/crop-doctor-offline' not in sys.path:
    sys.path.insert(0, '/content/crop-doctor-offline')
print("Repo ready.")

In [ ]:
# ── 4. Train Vision Model (MobileNetV3-Large + Cosine Annealing + Augmentations) ──
!python /content/crop-doctor-offline/ml/train_vision.py \
    --data_dir /content/crop_data \
    --class_map /content/crop-doctor-offline/data/class_map.yaml \
    --output_dir /content/drive/MyDrive/crop_doctor_runs \
    --model_name mobilenet_v3_large \
    --img_size 320 \
    --multi_task \
    --batch_size 64 \
    --epochs 15 \
    --lr 3e-4 \
    --device cuda \
    --export

In [ ]:
# ── 5. Verify ExecuTorch Export & Benchmark Rules H15 / H2 ────────────────────
import os, json
from IPython.display import Image, display

out_dir = '/content/drive/MyDrive/crop_doctor_runs'
pte_file = os.path.join(out_dir, 'vision.pte')
metrics_file = os.path.join(out_dir, 'metrics.json')
cm_file = os.path.join(out_dir, 'confusion_matrix.png')

if os.path.exists(pte_file):
    size_mb = os.path.getsize(pte_file) / (1024 * 1024)
    print(f"\n✓ Exported ExecuTorch Model: {pte_file}")
    print(f"✓ File Size: {size_mb:.2f} MB")
    print(f"✓ Rule H15 (<= 30 MB for 6 GB RAM Phone): {'PASS' if size_mb <= 30 else 'FAIL'}")
    print(f"✓ Rule H2  (<= 90 MB for GitHub commit):  {'PASS' if size_mb <= 90 else 'FAIL'}")
else:
    print(f"[ERROR] vision.pte was not found in {out_dir}")

if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        m = json.load(f)
    print(f"\n=== Accuracy Summary ===")
    print(f"Overall Top-1 Val Accuracy: {m.get('overall_top1_accuracy')}%")
    print(f"Real Field Holdout Accuracy: {m.get('real_field_accuracy')}%")
    print(f"Temperature Calibration T:   {m.get('temperature_T')}")
    print(f"Per-Crop Accuracy Breakdown: {m.get('per_crop_accuracy')}")

if os.path.exists(cm_file):
    print("\n=== Confusion Matrix ===")
    display(Image(cm_file))

## 6. How to Deploy to Your Android App

1. Download `vision.pte` from your Google Drive `crop_doctor_runs/vision.pte` to your laptop.
2. Copy `vision.pte` into your project repository at:
   `app/src/main/assets/models/vision.pte`
3. Rebuild the app in terminal or Android Studio:
   ```powershell
   ./gradlew assembleDebug
   ```
4. Install the APK on your emulator or connected Arm phone (`adb install -r app/build/outputs/apk/debug/app-debug.apk`).
   The app will automatically detect `vision.pte` on startup and run 100% offline vision inference!